# hexaload — Timestral-209M + LoRA fine-tuning

Zero-shot vs LoRA-fine-tuned `mistralai/Timestral-209M` on the held-out last 720 h of the eco2mix series, scored with the benchmark metrics (MAE/RMSE/MASE/CRPS) against RTE `prevision_j1`. 


In [ ]:
import os
from pathlib import Path

import pandas as pd
import torch
from safetensors.torch import save_file

from hexaload.backtest.load import exogenous_columns, load_features, load_targets
from hexaload.metrics.core import crps, mae, mase, rmse, sample_quantiles
from hexaload.models import DEFAULT_QUANTILES, TimestralLoRA, TimestralTSFM
from hexaload.progress import setup_logging

# set before the first model load (timestral/huggingface_hub are imported lazily)
os.environ.setdefault("HF_HUB_OFFLINE", "1")  # weights are in the local HF cache

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)  # hexaload loaders use repo-root-relative paths

setup_logging(verbose=True)

SEED = 42
torch.manual_seed(SEED)
REPO_ID = "mistralai/Timestral-209M"  # "mistralai/Timestral-28M" for a fast smoke test
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
CTX_HOURS = 24 * 365  # eval context, like the backtest zero-shot
TEST_HOURS = 720  # held-out block = last 30 days
D1_HOURS = 24  # fair D-1 comparison block
LORA = dict(
    lora_r=8,
    lora_alpha=16.0,
    lr=5e-5,
    n_steps=300,
    batch=2,
    train_context_hours=24 * 30,
    seed=SEED,
)
OUT_DIR = Path("data/notebooks")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| repo:", REPO_ID)

In [ ]:
y, ref, wx, _ = load_targets()
X, manifest = load_features()
exo = exogenous_columns(manifest)
y = y.interpolate(method="time").bfill().ffill()
X_exo = X[exo].reindex(y.index).interpolate(method="time").bfill().ffill().fillna(0.0)

cutoff = y.index.max() - pd.Timedelta(hours=TEST_HOURS)
in_tr = y.index <= cutoff
y_train, y_test = y[in_tr], y[~in_tr]
X_tr, X_te = X_exo[in_tr], X_exo[~in_tr]
test_idx = y_test.index
print(
    f"train {y_train.index.min():%Y-%m-%d} -> {y_train.index.max():%Y-%m-%d} | "
    f"test {test_idx.min():%Y-%m-%d} -> {test_idx.max():%Y-%m-%d} | exo {len(exo)} cols"
)

In [ ]:
def score(pred: pd.Series, qdf: pd.DataFrame, y_true: pd.Series) -> dict:
    samples = sample_quantiles(qdf, DEFAULT_QUANTILES, n_samples=200, seed=0)
    return {
        "mae": mae(y_true, pred),
        "rmse": rmse(y_true, pred),
        "mase": mase(y_true, pred, y_train),
        "crps": crps(y_true, samples),
    }

## Zero-shot — `TimestralTSFM`

On MPS the float64→float32 shim (`models/_mps_shim.py`) is applied automatically at load; the cell below checks it against the CPU path.


In [ ]:
m0 = TimestralTSFM(REPO_ID, device=DEVICE, context_hours=CTX_HOURS).fit(y_train, X_tr)
pred0 = m0.predict(test_idx, X_te)
qdf0 = m0.predict_quantiles(test_idx, X_te, DEFAULT_QUANTILES)
pred0_d1 = m0.predict(test_idx[:D1_HOURS], X_te.iloc[:D1_HOURS])
qdf0_d1 = m0.predict_quantiles(
    test_idx[:D1_HOURS], X_te.iloc[:D1_HOURS], DEFAULT_QUANTILES
)
metrics0 = score(pred0, qdf0, y_test)
metrics0_d1 = score(pred0_d1, qdf0_d1, y_test.iloc[:D1_HOURS])
print("zero-shot D+30:", {k: round(v, 1) for k, v in metrics0.items()})
print("zero-shot D-1 :", {k: round(v, 1) for k, v in metrics0_d1.items()})

if DEVICE == "mps":
    m_cpu = TimestralTSFM(REPO_ID, device="cpu", context_hours=CTX_HOURS).fit(
        y_train, X_tr
    )
    d = float(
        (m_cpu.predict(test_idx[:D1_HOURS], X_te.iloc[:D1_HOURS]) - pred0_d1)
        .abs()
        .max()
    )
    print(f"shim check: max |CPU - MPS| on D-1 median = {d:.4f} MW")
    del m_cpu
    torch.mps.empty_cache()

## LoRA fine-tune — `TimestralLoRA`

The whole training loop lives in the repo (`models/tsfm_lora.py`): adapters injected in the frozen base, pinball loss on the native quantiles via `forecast.__wrapped__`, windows sampled from the train split only.


In [ ]:
m1 = TimestralLoRA(REPO_ID, device=DEVICE, context_hours=CTX_HOURS, **LORA).fit(
    y_train, X_tr
)
if DEVICE == "mps":
    torch.mps.empty_cache()  # free training activations before the eval forecasts
n_train = sum(p.numel() for p in m1._pipe.model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in m1._pipe.model.parameters())
print(f"trainable: {n_train:,} / {n_all:,} ({100 * n_train / n_all:.2f}%)")

pred1 = m1.predict(test_idx, X_te)
qdf1 = m1.predict_quantiles(test_idx, X_te, DEFAULT_QUANTILES)
pred1_d1 = m1.predict(test_idx[:D1_HOURS], X_te.iloc[:D1_HOURS])
qdf1_d1 = m1.predict_quantiles(
    test_idx[:D1_HOURS], X_te.iloc[:D1_HOURS], DEFAULT_QUANTILES
)
metrics1 = score(pred1, qdf1, y_test)
metrics1_d1 = score(pred1_d1, qdf1_d1, y_test.iloc[:D1_HOURS])

lora_sd = {
    k: v.detach().cpu() for k, v in m1._pipe.model.state_dict().items() if "lora_" in k
}
adapter_path = OUT_DIR / "timestral_lora_adapter.safetensors"
save_file(lora_sd, str(adapter_path))
print(f"adapter saved: {len(lora_sd)} tensors -> {adapter_path}")

## Results — D-1 (h=24) is the fair row vs RTE (refreshed daily); the 720 h curve compounds block-AR drift, same rule as the D+7 table.


In [ ]:
name = REPO_ID.split("/")[-1].lower()
rte = ref.loc[test_idx]
y_d1 = y_test.iloc[:D1_HOURS]


def pts(pred, y_true):
    return {
        "mae": mae(y_true, pred),
        "rmse": rmse(y_true, pred),
        "mase": mase(y_true, pred, y_train),
        "crps": float("nan"),
    }


table = pd.DataFrame(
    {
        f"{name} zero-shot D-1": metrics0_d1,
        f"{name} lora D-1": metrics1_d1,
        "rte prevision_j1 D-1": pts(rte.iloc[:D1_HOURS], y_d1),
        f"{name} zero-shot D+30": metrics0,
        f"{name} lora D+30": metrics1,
        "rte prevision_j1 D+30": pts(rte, y_test),
    }
).T
print(table.round(2))
(OUT_DIR / "timestral_lora_metrics.json").write_text(table.round(4).to_json())

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

import hexaload.viz  # noqa: F401 — hexaload plotly template

pio.renderers.default = "notebook"

h = min(24 * 7, len(test_idx))
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=test_idx[:h],
        y=y_test.iloc[:h],
        name="actual",
        line=dict(color="black", width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=test_idx[:h],
        y=rte.iloc[:h],
        name="RTE prevision_j1",
        line=dict(color="green", dash="dot"),
    )
)
fig.add_trace(
    go.Scatter(
        x=test_idx[:h],
        y=pred0.iloc[:h],
        name="zero-shot",
        line=dict(color="gray", dash="dash"),
    )
)
fig.add_trace(
    go.Scatter(
        x=test_idx[:h], y=qdf1["q0.95"].iloc[:h], line=dict(width=0), showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x=test_idx[:h],
        y=qdf1["q0.05"].iloc[:h],
        name="lora q05-q95",
        fill="tonexty",
        fillcolor="rgba(214,39,40,0.15)",
        line=dict(width=0),
    )
)
fig.add_trace(
    go.Scatter(x=test_idx[:h], y=pred1.iloc[:h], name="lora", line=dict(color="red"))
)
fig.update_layout(
    title="held-out month, first 7 days (D+30 curve)",
    xaxis_title="UTC",
    yaxis_title="load (MW)",
)
fig.write_html(OUT_DIR / "timestral_lora_forecast.html")
fig.show()